# Problem 014:  Longest Collatz Sequence

> The following iterative sequence is defined for the set of positive integers:
>
> > $n \to n/2$ ($n$ is even)
> > 
> > $n \to 3n + 1$ ($n$ is odd)
>
> Using the rule above and starting with $13$, we generate the following sequence:
>
> $$13 \to 40 \to 20 \to 10 \to 5 \to 16 \to 8 \to 4 \to 2 \to 1.$$
>
> It can be seen that this sequence (starting at $13$ and finishing at $1$) contains $10$ terms. Although it has not been proved yet (Collatz Problem), it is thought that all starting numbers finish at $1$.
> 
> Which starting number, under one million, produces the longest chain?
> 
> NOTE: Once the chain starts the terms are allowed to go above one million.


## Naive approach $\mathcal O\left(N \ln N\right)$

In this approach, it is just a matter of coding the Collatz behavior and then running it for every value up to the max $N$.  A lot of values are being calculated multiple times, but it requires $\mathcal O (1)$ space to run.  One small optimization I introduce is that every odd $n$ will be followed by an even number.  Thus, two steps can be taken immediately for an odd value.

The time scaling is $\mathcal O (N\ln N)$ since there are on average $\mathcal O(\ln n)$ steps the Collatz sequence of $n$.

In [124]:
%%time

def p014_naive(N: int) -> int:
    val = (0,0)
    for i in range(1, N + 1):
        n = 1
        j = i
        while j != 1:
            if j % 2:
                n += 2
                j = (3 * j + 1) // 2
            else:
                n += 1
                j //= 2
        val = max(val, (n,i))
    return val

N = 1_000_000
ans = p014_naive(N)

print(ans)

(525, 837799)
CPU times: user 15.9 s, sys: 22.3 ms, total: 16 s
Wall time: 16.4 s


## Recursion with a cache approach $\mathcal O\left(N \ln N\right)$

One approach to cut down on repeated calculations is to write a recursive routine and decorate it with a `cache` function.

This obviously requires more space, but run $\sim 7$ times faster than the naive approach above.  I am not convinced the amount of space needed would be $\mathcal O(N)$ because the cache would also store any number larger than $N$ that it encounters, as well.

The time scaling will follow the same as the space scaling.  As such, the $\mathcal O(N\ln N)$ stated above is uncertain.

In [120]:
%%time

from functools import cache

@cache
def collatz(n: int) -> int:
    if n == 1:
        return 1
    if n % 2:
        return 2 + collatz((3 * n + 1) // 2)
    else:
        return 1 + collatz(n // 2)
    
def p014_recursive(N: int) -> int:
    val = (0,0)
    for i in range(1, N + 1):
        val = max(val, (collatz(i),i))
    return val

N = 1_000_000
ans = p014_recursive(N)

print(ans)

(525, 837799)
CPU times: user 2.23 s, sys: 231 ms, total: 2.46 s
Wall time: 2.55 s


This line shows how calculating the Collatz sequence of the first million values collects the stopping time for $1,588,207$ values.

In [121]:
collatz.cache_info()

CacheInfo(hits=999999, misses=1588207, maxsize=None, currsize=1588207)

## Non-resursive with memoization $\mathcal O\left(N \ln N\right)$

Here is an alternative approach to reducing the number of calculations needed that turns out to be faster than recursion.  I chose to make an array that acts as a cache, or memoization, that only stores the values of the stopping time for values less than the tested value.  The storage is then scales strictly as $\mathcal O(N)$.  There are more calculations being performed, since any value larger than $n$ is not stored in the array, but there are less checks to the cache and linear code runs faster than recursive code in general.  I see a factor of $2$ increase in speed in this way.

The time scaling I quote above is questionable in the same way as the recursive algorithm, where the "dropping time" of the sequence is of questionable number of steps.

In [123]:
%%time

def p014_memo(N: int) -> int:
    s = [None] * (N+1)
    s[1] = 1
    val = (0,0)

    for i in range(2, N + 1):
        n = 0
        j = i
        while j >= i:
            if j % 2:
                n += 2
                j = (3 * j + 1) // 2
            else:
                n += 1
                j //= 2
                
        n += s[j]
        s[i] = n
        #print(n,i)
        val = max(val, (n,i))
    return val[1]

N = 1_000_000
ans = p014_memo(N)

print(ans)

837799
CPU times: user 1.21 s, sys: 3.97 ms, total: 1.21 s
Wall time: 1.25 s
